In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np

app = FastAPI()

# Load model (assuming files are in same folder)
model = joblib.load('heart_disease_model.pkl')
scaler = joblib.load('scaler.pkl')
imputer = joblib.load('imputer.pkl')
feature_names = joblib.load('feature_names.pkl')

class Patient(BaseModel):
    Age: int
    Sex: int
    Chest_Pain_Type: int
    Resting_Blood_Pressure: int
    Cholesterol: int
    Fasting_Blood_Sugar: int
    Resting_ECG: int
    Max_Heart_Rate: int
    Exercise_Induced_Angina: int
    ST_Depression: float
    Slope_of_ST: int
    Number_of_Major_Vessels: int
    Thalassemia: int

@app.get("/")
def root():
    return {"message": "Heart Disease Prediction API", "status": "ready"}

@app.post("/predict")
def predict(patient: Patient):
    # Prepare data
    data = {
        'Age': patient.Age,
        'Sex': patient.Sex,
        'Chest Pain Type': patient.Chest_Pain_Type,
        'Resting Blood Pressure': patient.Resting_Blood_Pressure,
        'Cholesterol': patient.Cholesterol,
        'Fasting Blood Sugar': patient.Fasting_Blood_Sugar,
        'Resting ECG': patient.Resting_ECG,
        'Max Heart Rate': patient.Max_Heart_Rate,
        'Exercise-Induced Angina': patient.Exercise_Induced_Angina,
        'ST Depression': patient.ST_Depression,
        'Slope of ST Segment': patient.Slope_of_ST,
        'Number of Major Vessels': patient.Number_of_Major_Vessels,
        'Thalassemia': patient.Thalassemia
    }
    
    df = pd.DataFrame([data])
    
    # Add features
    df['Cholesterol_Risk'] = (df['Cholesterol'] > 240).astype(int)
    df['BP_Risk'] = (df['Resting Blood Pressure'] > 140).astype(int)
    df['HR_Risk'] = (df['Max Heart Rate'] < 120).astype(int)
    
    df = df[feature_names]
    scaled = scaler.transform(df)
    prob = model.predict_proba(scaled)[0][1]
    pred = model.predict(scaled)[0]
    
    return {
        "has_heart_disease": bool(pred),
        "probability": round(prob * 100, 2),
        "risk": "HIGH" if prob > 0.7 else "MODERATE" if prob > 0.3 else "LOW"
    }